# TransMilenio · preparación y análisis exploratorio

## 1. Pregunta de investigación

> ¿En qué estaciones y franjas horarias se concentran más validaciones respecto a los pasos programados y, donde haya suficientes posiciones GPS, qué tan cerca coincide el paso estimado de los buses con el horario previsto?

El piloto explora las validaciones del 15 y 16 de noviembre y la oferta programada del 18 al 23. Medir la relación entre ambas requiere pares de la misma fecha, hora y estación. Evaluar el paso estimado frente al horario requiere capturas GPS suficientes y vinculadas al GTFS.

## 2. Carga y unidad de análisis
Cada fila de la base principal representa un código de estación, fecha civil y hora de Bogotá. Una validación registra un acceso. Por su parte, la oferta cuenta eventos programados para una estación y hora. La dimensión contiene una fila por código y los atributos socioeconómicos describen su localidad.

In [1]:
from pathlib import Path
import pandas as pd

raiz = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
necesarios = ['base_dashboard','entradas_estacion_hora','oferta_programada_estacion_hora',
              'estaciones_localidad_poblacion','auditoria_validaciones','auditoria_gtfs',
              'auditoria_cruce_fecha','auditoria_faltantes_base','inventario_insumos']
procesados = raiz / 'datos' / 'procesados'
datos = procesados if all((procesados / f'{n}.csv').is_file() for n in necesarios) else raiz / 'datos' / 'demo'
tipos = {'codigo_estacion':'string','codigo_localidad':'string'}
tablas = {n:pd.read_csv(datos / f'{n}.csv',dtype=tipos) for n in necesarios}
base, entradas, oferta, estaciones = [tablas[n] for n in necesarios[:4]]
llave = ['fecha','hora','codigo_estacion']
print(datos.relative_to(raiz))
print({n:t.shape for n,t in tablas.items()})

datos/demo
{'base_dashboard': (5546, 20), 'entradas_estacion_hora': (5633, 8), 'oferta_programada_estacion_hora': (17511, 7), 'estaciones_localidad_poblacion': (178, 11), 'auditoria_validaciones': (2, 7), 'auditoria_gtfs': (6, 10), 'auditoria_cruce_fecha': (8, 4), 'auditoria_faltantes_base': (20, 3), 'inventario_insumos': (11, 3)}


## 3. Diagnóstico inicial
Las auditorías provienen de leer los ZIP originales por bloques. La muestra permite reproducir el diagnóstico sin redistribuir registros individuales.

In [2]:
print(base[llave + ['entradas_registradas','localidad']].head().to_string(index=False))
base.info()
print('Faltantes más frecuentes:')
print(base.isna().sum().nlargest(10).to_string())
print('Duplicados por estación, fecha y hora:', base.duplicated(llave).sum())
print(tablas['auditoria_validaciones'].to_string(index=False))
print(tablas['auditoria_gtfs'][['fecha_servicio','viajes_troncales_activos','eventos_programados']].to_string(index=False))

     fecha  hora codigo_estacion  entradas_registradas localidad
2026-11-15     3           02304                     2 CHAPINERO
2026-11-15     3           05000                   105   KENNEDY
2026-11-15     3           05001                     3   KENNEDY
2026-11-15     3           05002                     1   KENNEDY
2026-11-15     3           05102                     1   KENNEDY
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5546 entries, 0 to 5545
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   fecha                           5546 non-null   object 
 1   hora                            5546 non-null   int64  
 2   codigo_estacion                 5546 non-null   string 
 3   entradas_registradas            5546 non-null   int64  
 4   nombre                          5546 non-null   object 
 5   tipo_punto                      5546 non-null   object 
 6   dia_semana

## 4. Limpieza, integración y controles
El script de src selecciona columnas, filtra observaciones, estandariza tipos y códigos, agrupa y ordena. Los tres códigos llamados Corral o Bicicletero permanecen en la tabla de control. La oferta usa el calendario GTFS y agrupa plataformas en la estación principal. Se conservan los faltantes de localidad y oferta. La bitácora en docs explica las decisiones y los comandos.

In [3]:
especiales = entradas.loc[entradas.tipo_punto.eq('Punto especial'),'entradas_registradas'].sum()
assert entradas.entradas_registradas.sum() == base.entradas_registradas.sum() + especiales
assert not base.duplicated(llave).any() and not oferta.duplicated(llave).any()
assert estaciones.codigo_estacion.is_unique
assert base.hora.between(0,23).all() and oferta.hora.between(0,23).all()
assert oferta.paradas_programadas.sum() == tablas['auditoria_gtfs'].eventos_programados.sum()
assert tablas['auditoria_cruce_fecha'].pares_con_oferta.sum() == base.paradas_programadas.notna().sum()
print(tablas['auditoria_cruce_fecha'].to_string(index=False))
print(base.groupby('estado_territorio').entradas_registradas.sum().to_string())
print('Validaciones de puntos especiales:', especiales)

     fecha  validaciones  gtfs_de_fecha_servicio  pares_con_oferta
2026-11-15          True                   False                 0
2026-11-16          True                   False                 0
2026-11-18         False                    True                 0
2026-11-19         False                    True                 0
2026-11-20         False                    True                 0
2026-11-21         False                    True                 0
2026-11-22         False                    True                 0
2026-11-23         False                    True                 0
estado_territorio
Completo                          1002828
Código ausente en capa oficial      67250
Sin localidad asignada              63466
Validaciones de puntos especiales: 22809


## 5. Análisis exploratorio
Primero se observa la distribución de entradas por hora. Después se ordenan estaciones y localidades para identificar concentraciones. Los conteos territoriales solo incluyen estaciones con localidad asignada. La población de una localidad se cuenta una sola vez.

In [4]:
por_hora = base.groupby(['fecha','hora'],as_index=False).entradas_registradas.sum()
picos = por_hora.sort_values(['fecha','entradas_registradas','hora'],ascending=[True,False,True]).drop_duplicates('fecha')
por_estacion = base.groupby(['codigo_estacion','nombre']).entradas_registradas.sum().nlargest(8)
por_localidad = base.dropna(subset=['localidad']).groupby('localidad').entradas_registradas.sum().nlargest(5)
print('Picos por fecha:'); print(picos.to_string(index=False))
print('Estaciones:'); print(por_estacion.to_string())
print('Localidades con ubicación verificada:'); print(por_localidad.to_string())

Picos por fecha:
     fecha  hora  entradas_registradas
2026-11-15    11                 46088
2026-11-16    17                 42767
Estaciones:
codigo_estacion  nombre                         
02000            Portal Norte - Unicervantes        61626
05000            Portal Américas                    45908
03000            Portal Suba                        42847
07000            Portal Sur JFK Coop. Financiera    40639
04000            Cabecera Calle 80                  39579
09000            Cabecera Usme                      35525
07503            SAN MATEO - C.C. UNISUR            35134
10000            Portal 20 de Julio                 34933
Localidades con ubicación verificada:
localidad
USAQUEN          132259
KENNEDY          107365
ENGATIVA          85570
PUENTE ARANDA     74907
SUBA              74776


## 6. Hallazgos, límites y siguiente paso
Portal Norte registró 61.626 entradas entre los dos días. El pico agregado fue a las 11:00 el domingo y a las 17:00 el lunes festivo. Entre las estaciones con localidad asignada, Usaquén concentró 132.259 entradas. La muestra no permite generalizar a días laborables ni atribuir el cambio de pico a una causa. Para contrastar demanda y oferta hace falta conseguir validaciones y GTFS para una misma fecha. El piloto GPS necesita capturas consecutivas, tiempos frescos y un enlace validado con el horario.